# Acidification Profiles in Cheese Making

**Contact:** Dimitri Bocquel  
**Authors:** Lilandra Albert-Lavault & Samuele Moungang  
**Data collected:** 28 April 2026

| Source | File | Résolution | Canaux |
|--------|------|------------|--------|
| Fromage 1 — USB | `trace pH fromage 1_USB_28042026.csv` | 1 min | pH, mV |
| Fromage 2 — Server | `trace pH fromage 2_Server_28042026.csv` | ~1 s | pH |
| Fromage 3 — Hannah | `trace pH fromage 3_28042026_hannah.csv` | 1 min | pH, mV, T°C |

---

## Pipeline
1. **Load & inspect** raw data
2. **Interpolation** — combler les lacunes du signal server (grille 1 s régulière)
3. **Resampling** — grille 1 min commune + lissage Savitzky-Golay
4. **Correction Nernst** — corriger le pH mesuré en fonction de la température réelle de l'électrode

---
## 0. Setup

In [ ]:
import sys
# Décommentez si scipy est absent :
# !{sys.executable} -m pip install scipy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d

plt.rcParams.update({
    'figure.figsize': (14, 4.5),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Constantes physiques — équation de Nernst
R    = 8.314      # J mol⁻¹ K⁻¹
F    = 96485.0    # C mol⁻¹
LN10 = np.log(10)

def nernst_slope(T_celsius):
    """Pente de Nernst théorique en mV/pH à la température T (°C)."""
    return -(R * (T_celsius + 273.15) * LN10 / F) * 1000

T_CAL = 25.0   # Température de calibration de référence supposée (USB & Server)
print(f'Pente Nernst @ {T_CAL}°C : {nernst_slope(T_CAL):.3f} mV/pH')
print(f'Pente Nernst @ 37°C    : {nernst_slope(37):.3f} mV/pH')

---
## 1. Chargement & Inspection des Données Brutes

In [ ]:
# ── Fromage 1 — USB (pH + mV, 1 min) ─────────────────────────────────────────
import sys
!{sys.executable} -m pip install scipy
= pd.read_csv(
    'data/trace pH fromage 1_USB_28042026.csv',
    usecols=['Date', 'Time', 'Ch1_M1', 'Ch1_M3'],
)
df1_raw['datetime'] = pd.to_datetime(
    df1_raw['Date'] + ' ' + df1_raw['Time'], format='%d/%b/%Y %H:%M:%S'
)
df1_raw['pH'] = pd.to_numeric(df1_raw['Ch1_M1'], errors='coerce')
df1_raw['mV']  = pd.to_numeric(df1_raw['Ch1_M3'],  errors='coerce')
df1 = df1_raw[['datetime', 'pH', 'mV']].dropna().set_index('datetime').sort_index()
print(f'Fromage 1 (USB)    : {len(df1):,} points | {df1.index[0]} → {df1.index[-1]}')
df1.describe()

In [ ]:
# ── Fromage 2 — Server (pH uniquement, ~1 s) ──────────────────────────────────
df2_raw = pd.read_csv(
    'data/trace pH fromage 2_Server_28042026.csv',
    names=['datetime', 'pH'], skiprows=1,
)
df2_raw['datetime'] = pd.to_datetime(df2_raw['datetime'])
df2_raw['pH'] = pd.to_numeric(df2_raw['pH'], errors='coerce')
df2_raw = df2_raw.dropna().set_index('datetime').sort_index()
# Valeurs physiquement plausibles uniquement
df2 = df2_raw[(df2_raw['pH'] >= 3.5) & (df2_raw['pH'] <= 8.0)].copy()
print(f'Fromage 2 (Server) : {len(df2):,} points | {df2.index[0]} → {df2.index[-1]}')
df2.describe()

In [ ]:
# ── Fromage 3 — Hannah (pH + mV + T°C, 1 min) ────────────────────────────────
# 17 lignes de métadonnées à ignorer
df3_raw = pd.read_csv(
    'data/trace pH fromage 3_28042026_hannah.csv',
    skiprows=17, header=0,
    names=['_', 'rec', 'date', 'time', 'pH', 'u1', 'mV', 'u2', 'temp_C', 'u3'],
    usecols=['date', 'time', 'pH', 'mV', 'temp_C'],
    encoding='latin-1',
)
df3_raw['datetime'] = pd.to_datetime(df3_raw['date'] + ' ' + df3_raw['time'])
for col in ['pH', 'mV', 'temp_C']:
    df3_raw[col] = pd.to_numeric(df3_raw[col], errors='coerce')
df3 = df3_raw[['datetime', 'pH', 'mV', 'temp_C']].dropna().set_index('datetime').sort_index()
print(f'Fromage 3 (Hannah) : {len(df3):,} points | {df3.index[0]} → {df3.index[-1]}')
df3.describe()

In [ ]:
# Vue d'ensemble brute — 3 panneaux
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fmt = mdates.DateFormatter('%H:%M')
loc = mdates.HourLocator(interval=2)

for ax, df, label, color in zip(
    axes,
    [df1, df2, df3],
    ['Fromage 1 (USB)', 'Fromage 2 (Server, brut)', 'Fromage 3 (Hannah)'],
    ['steelblue', 'tomato', 'seagreen'],
):
    ax.plot(df.index, df['pH'], color=color, linewidth=0.8, alpha=0.85)
    ax.xaxis.set_major_formatter(fmt)
    ax.xaxis.set_major_locator(loc)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    ax.set_title(label)
    ax.set_ylabel('pH')
    ax.set_xlabel('Heure (28 avr. 2026)')

plt.suptitle('Traces pH brutes — trois fromages', fontweight='bold')
plt.tight_layout()
plt.savefig('01_raw_traces.png', bbox_inches='tight')
plt.show()

---
## 2. Interpolation — Combler les Lacunes du Signal Server (Fromage 2)

Le logger server enregistre à des intervalles irréguliers (~1 s) avec des coupures.  
On construit une grille temporelle régulière à **1 seconde** et on comble les lacunes
par **interpolation linéaire**. Les segments dont la lacune dépasse 30 s sont marqués
comme non fiables.

In [ ]:
# ── Détection des lacunes ─────────────────────────────────────────────────────
GAP_THRESHOLD_S = 30
dt_seconds = df2.index.to_series().diff().dt.total_seconds()
gaps = dt_seconds[dt_seconds > GAP_THRESHOLD_S].dropna()
print(f'Lacunes > {GAP_THRESHOLD_S} s dans Fromage 2 : {len(gaps)}')
if len(gaps):
    print(gaps.apply(lambda s: f'{s:.0f} s').to_frame('durée_lacune'))

# ── Grille régulière 1 s ──────────────────────────────────────────────────────
t_start    = df2.index[0]
t_end      = df2.index[-1]
regular_idx = pd.date_range(t_start, t_end, freq='1s')

t_orig = (df2.index - t_start).total_seconds().values
t_new  = (regular_idx - t_start).total_seconds().values

interp_fn  = interp1d(t_orig, df2['pH'].values, kind='linear',
                      bounds_error=False, fill_value=np.nan)
pH_interp  = interp_fn(t_new)

# ── Masquer les segments interpolés dans les grandes lacunes ──────────────────
reliable = np.ones(len(regular_idx), dtype=bool)
for gap_ts, gap_dur in gaps.items():
    prev_ts = gap_ts - pd.Timedelta(seconds=gap_dur)
    mask = (regular_idx > prev_ts) & (regular_idx < gap_ts)
    reliable[mask] = False

df2_interp = pd.DataFrame({'pH': pH_interp, 'reliable': reliable}, index=regular_idx)
print(f'\nGrille régulière : {len(df2_interp):,} pts | fiables : {reliable.sum():,}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

# Zones d'interpolation non fiables
in_gap, gap_start = False, None
for ts, row in df2_interp.iterrows():
    if not row['reliable'] and not in_gap:
        gap_start, in_gap = ts, True
    elif row['reliable'] and in_gap:
        ax.axvspan(gap_start, ts, alpha=0.2, color='orange',
                   label='Interpolé (lacune non fiable)')
        in_gap = False
if in_gap:
    ax.axvspan(gap_start, df2_interp.index[-1], alpha=0.2, color='orange')

ax.plot(df2.index, df2['pH'],
        color='grey', linewidth=0.5, alpha=0.5, label='Brut (points originaux)')
ax.plot(df2_interp.index, df2_interp['pH'],
        color='tomato', linewidth=0.8, label='Interpolé (grille 1 s)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
ax.set_title('Fromage 2 (Server) — Interpolation linéaire sur grille 1 s', fontweight='bold')
ax.set_xlabel('Heure (28 avr. 2026)')
ax.set_ylabel('pH')
handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys())
plt.tight_layout()
plt.savefig('02_interpolation.png', bbox_inches='tight')
plt.show()

---
## 3. Rééchantillonnage — Grille 1 min Commune + Lissage

Les trois signaux sont rééchantillonnés à la **médiane sur 1 minute** pour :
- supprimer le bruit sous-minute (surtout le signal server)
- aligner les horodatages pour la comparaison

Un filtre **Savitzky-Golay** (fenêtre = 11 min, degré = 3) est ensuite appliqué
pour le calcul propre des dérivées.

In [ ]:
RESAMPLE_FREQ = '1min'
SG_WINDOW     = 11   # doit être impair
SG_POLY       = 3

def resample_and_smooth(series, freq=RESAMPLE_FREQ, window=SG_WINDOW, poly=SG_POLY):
    resampled = series.resample(freq).median().dropna()
    if len(resampled) < window:
        return resampled, resampled.copy()
    smoothed = savgol_filter(resampled.values, window_length=window, polyorder=poly)
    return resampled, pd.Series(smoothed, index=resampled.index, name=series.name)

df1_res, df1_smooth = resample_and_smooth(df1['pH'])
df2_res, df2_smooth = resample_and_smooth(df2_interp['pH'])   # signal interpolé
df3_res, df3_smooth = resample_and_smooth(df3['pH'])

# Rééchantillonnage mV et température
df1_mV_res   = df1['mV'].resample(RESAMPLE_FREQ).median().dropna()
df3_mV_res   = df3['mV'].resample(RESAMPLE_FREQ).median().dropna()
df3_temp_res = df3['temp_C'].resample(RESAMPLE_FREQ).median().dropna()

print('Tailles après rééchantillonnage :')
for name, s in [('Fromage 1', df1_res), ('Fromage 2', df2_res), ('Fromage 3', df3_res)]:
    print(f'  {name} : {len(s):,} points')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
fmt = mdates.DateFormatter('%H:%M')
loc = mdates.HourLocator(interval=2)

datasets = [
    (df1_res, df1_smooth, 'Fromage 1 (USB)',    'steelblue'),
    (df2_res, df2_smooth, 'Fromage 2 (Server)', 'tomato'),
    (df3_res, df3_smooth, 'Fromage 3 (Hannah)', 'seagreen'),
]

for ax, (res, smooth, label, color) in zip(axes, datasets):
    ax.plot(res.index,    res.values,    color='lightgrey', linewidth=1.0, label='Médiane 1 min')
    ax.plot(smooth.index, smooth.values, color=color,       linewidth=1.8, label='Savitzky-Golay')
    ax.xaxis.set_major_formatter(fmt)
    ax.xaxis.set_major_locator(loc)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    ax.set_title(label)
    ax.set_ylabel('pH')
    ax.set_xlabel('Heure (28 avr. 2026)')
    ax.legend(fontsize=9)

plt.suptitle('Étape 3 — Rééchantillonnage & Lissage (médiane 1 min + Savitzky-Golay)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('03_resampling.png', bbox_inches='tight')
plt.show()

In [ ]:
# Profil de température — Fromage 3
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(df3_temp_res.index, df3_temp_res.values, color='darkorange', linewidth=1.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
ax.set_title('Fromage 3 (Hannah) — Profil de température', fontweight='bold')
ax.set_ylabel('Température (°C)')
ax.set_xlabel('Heure (28 avr. 2026)')
plt.tight_layout()
plt.show()

---
## 4. Correction de Nernst — Correction du pH par la Température

Une électrode pH suit l'**équation de Nernst** :

$$E = E_0 - \frac{RT\ln 10}{F}\cdot\mathrm{pH}$$

La pente $S(T) = -\frac{RT\ln 10}{F}$ dépend de la température
(-59,16 mV/pH à 25 °C, -61,54 mV/pH à 37 °C).  
Lorsque la calibration se fait à $T_{\text{cal}}$ mais la mesure à $T_{\text{mes}}$,
le pH affiché est décalé. La valeur corrigée est :

$$\mathrm{pH}_{\text{corr}} =
  \mathrm{pH}_{\text{iso}} +
  (\mathrm{pH}_{\text{mes}} - \mathrm{pH}_{\text{iso}})
  \cdot \frac{T_{\text{mes}}}{T_{\text{cal}}}$$

où $T$ est en **Kelvin** et $\mathrm{pH}_{\text{iso}} \approx 7{,}0$ (point isoélectrique).

| Source | T disponible ? | mV disponible ? | Correction |
|--------|----------------|-----------------|------------|
| Fromage 1 (USB)    | ✗ supposée 35 °C | ✓ | Rapport de pente |
| Fromage 2 (Server) | ✗ supposée 35 °C | ✗ | Rapport de pente |
| Fromage 3 (Hannah) | ✓ mesurée        | ✓ | Correction complète |

In [ ]:
PH_ISO  = 7.0    # point isoélectrique (typiquement 7 pour la plupart des électrodes)

def nernst_correct(pH_meas, T_meas_C, T_cal_C=T_CAL, pH_iso=PH_ISO):
    """
    Corrige le pH mesuré pour l'écart de température par rapport à la calibration.
    pH_meas et T_meas_C peuvent être des Series ou arrays.
    """
    T_meas_K = T_meas_C + 273.15
    T_cal_K  = T_cal_C  + 273.15
    return pH_iso + (pH_meas - pH_iso) * (T_meas_K / T_cal_K)

# ── Fromage 3 : correction complète avec la température mesurée ───────────────
idx3      = df3_smooth.index.intersection(df3_temp_res.index)
pH3_meas  = df3_smooth.loc[idx3]
temp3     = df3_temp_res.loc[idx3]
pH3_corr  = pd.Series(
    nernst_correct(pH3_meas.values, temp3.values),
    index=idx3, name='pH_corr'
)

# ── Fromage 1 : température supposée 35 °C ────────────────────────────────────
T_F1_ASSUMED = 35.0
pH1_corr = nernst_correct(df1_smooth, T_meas_C=T_F1_ASSUMED)

# ── Fromage 2 : température supposée 35 °C ────────────────────────────────────
T_F2_ASSUMED = 35.0
pH2_corr = nernst_correct(df2_smooth, T_meas_C=T_F2_ASSUMED)

# Résumé des corrections
d1 = (pH1_corr - df1_smooth).describe()
d3 = (pH3_corr - pH3_meas).describe()
print('Δ pH (corrigé − mesuré) :')
pd.DataFrame({'Fromage 1 (35 °C supposés)': d1,
              'Fromage 3 (T mesurée)': d3})

In [ ]:
# ── Avant / après correction pour chaque fromage ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

spec = [
    (df1_smooth, pH1_corr, f'Fromage 1 (USB, T supposée {T_F1_ASSUMED} °C)', 'steelblue'),
    (df2_smooth, pH2_corr, f'Fromage 2 (Server, T supposée {T_F2_ASSUMED} °C)', 'tomato'),
    (pH3_meas,   pH3_corr,  'Fromage 3 (Hannah, T mesurée)', 'seagreen'),
]

for ax, (measured, corrected, label, color) in zip(axes, spec):
    ax.plot(measured.index,  measured.values,  color='lightgrey', linewidth=1.5,
            label='Avant correction')
    ax.plot(corrected.index, corrected.values, color=color, linewidth=1.8,
            label='Corrigé Nernst')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('pH')
    ax.set_xlabel('Heure (28 avr. 2026)')
    ax.legend(fontsize=9)

plt.suptitle('Étape 4 — Correction Nernst (température)', fontweight='bold')
plt.tight_layout()
plt.savefig('04_nernst_correction.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Fromage 3 : détail correction vs. fluctuations de température ─────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(temp3.index, temp3.values, color='darkorange', linewidth=1.4)
axes[0].set_ylabel('Température (°C)')
axes[0].set_title('Fromage 3 — Température & détail de la correction Nernst',
                  fontweight='bold')

axes[1].plot(pH3_meas.index, pH3_meas.values, color='lightgrey', linewidth=1.5,
             label='Mesuré')
axes[1].plot(pH3_corr.index, pH3_corr.values, color='seagreen', linewidth=1.8,
             label='Corrigé Nernst')
axes[1].fill_between(pH3_meas.index, pH3_meas.values, pH3_corr.values,
                     alpha=0.2, color='seagreen', label='Δ correction')
axes[1].set_ylabel('pH')
axes[1].set_xlabel('Heure (28 avr. 2026)')
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[1].xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('04b_nernst_fromage3_detail.png', bbox_inches='tight')
plt.show()

---
## 5. Comparaison Finale — Profils Corrigés (alignés depuis t = 0)

In [ ]:
def hours_from_start(series):
    return (series.index - series.index[0]).total_seconds() / 3600

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(hours_from_start(pH1_corr), pH1_corr.values,
        color='steelblue', linewidth=2.0, label=f'Fromage 1 (USB, {T_F1_ASSUMED} °C supposés)')
ax.plot(hours_from_start(pH2_corr), pH2_corr.values,
        color='tomato',    linewidth=2.0, label=f'Fromage 2 (Server, {T_F2_ASSUMED} °C supposés)')
ax.plot(hours_from_start(pH3_corr), pH3_corr.values,
        color='seagreen',  linewidth=2.0, label='Fromage 3 (Hannah, T mesurée)')

for thr, ls in [(5.2, '--'), (5.0, ':'), (4.8, '-.')]:
    ax.axhline(thr, color='black', linewidth=0.7, linestyle=ls,
               alpha=0.5, label=f'pH {thr}')

ax.set_xlabel('Temps depuis le début du batch (heures)')
ax.set_ylabel('pH (corrigé Nernst)')
ax.set_title('Profils d\'acidification — Corrigés Nernst — Comparaison alignée',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('05_final_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Table des jalons clés ──────────────────────────────────────────────────────
def first_crossing(series, threshold):
    below = series[series <= threshold]
    return below.index[0] if not below.empty else None

thresholds = [5.5, 5.2, 5.0, 4.8]
rows = []
for thr in thresholds:
    t1 = first_crossing(pH1_corr, thr)
    t2 = first_crossing(pH2_corr, thr)
    t3 = first_crossing(pH3_corr, thr)
    rows.append({
        'Seuil pH': thr,
        'Fromage 1': t1.strftime('%H:%M') if t1 else '—',
        'Fromage 2': t2.strftime('%H:%M') if t2 else '—',
        'Fromage 3': t3.strftime('%H:%M') if t3 else '—',
    })

pd.DataFrame(rows).set_index('Seuil pH')

---
## 6. Notes & Prochaines Étapes

- [ ] Confirmer la température réelle de calibration du logger USB (supposée **35 °C** ici)
- [ ] Confirmer le pH isoélectrique de chaque modèle de sonde
- [ ] Ajouter les scores sensoriels / texturaux dès qu'ils seront disponibles
- [ ] Corréler les pics de vitesse d'acidification avec les journaux de cultures starters
- [ ] Analyser le bruit alternatif dans le signal server Fromage 2